# 基于 Swin-Tiny 的图像分类微调

## 1. 案例简介

本案例展示了如何使用 MindSpore 和 MindSpore NLP 库，在自定义数据集（EuroSAT）上微调预训练的 Swin-Tiny 视觉 Transformer 模型。

**Swin Transformer** 是一种分层的 Vision Transformer，其通过移动窗口计算自注意力（Shifted Windows），在图像分类、目标检测等任务上取得了优异的性能。Swin-Tiny 是该系列中轻量级的版本，适合在算力有限或需要快速迭代的场景下使用。

### 任务流程
1. **环境准备**：安装并导入必要的库。
2. **数据加载**：加载 EuroSAT 卫星图像数据集。
3. **数据预处理**：使用 MindSpore Dataset 对图像进行 Resize、归一化等操作。
4. **模型构建**：加载预训练的 `swin-tiny-patch4-window7-224` 模型，并修改分类头。
5. **模型微调**：配置训练参数，使用 `Trainer` 接口进行训练。
6. **推理演示**：加载微调后的模型对新图像进行预测。

### 运行环境

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10   | 2.7.0     | 0.5.1  |

In [ ]:
#若在https://internstudio-ascend.intern-ai.org.cn/console/instance进行开发时，使用notebook会出现无法正常使用NPU，可进行以下步骤：
# 进入开发机的命令窗口
# 1. 激活你的环境
# conda activate mind_py310
# pip install ipykernel
# python -m ipykernel install --user --name=mind_py310 --display-name="Python (mind_py310)"
# 2. 加载系统基础驱动配置
# source /usr/local/Ascend/ascend-toolkit/set_env.sh
# 3.【核心步骤】手动补全深层驱动路径 (修复 libascend_hal.so 报错)
# export LD_LIBRARY_PATH=/usr/local/Ascend/driver/lib64/driver:/usr/local/Ascend/driver/lib64/common:/usr/local/Ascend/driver/lib64:$LD_LIBRARY_PATH
# 4. 启动 Jupyter Lab
#jupyter lab --allow-root

In [ ]:
# 安装依赖
# !pip install mindnlp==0.5.1
# !pip install scikit-learn

## 2. 环境配置
首先，我们需要应用一个补丁来修复 mindtorch 的版本兼容性问题，并配置 MindSpore 的运行环境。

In [ ]:
# --------兼容性补丁---------
import mindtorch.autograd.function
if not hasattr(mindtorch.autograd.function, 'FunctionCtx'):
    class FunctionCtx:
        def __init__(self):
            self.saved_tensors = ()
        def save_for_backward(self, *tensors):
            self.saved_tensors = tensors
    mindtorch.autograd.function.FunctionCtx = FunctionCtx
    print("已应用 mindtorch 兼容性修复")

# ----基础配置与环境 ----
import mindnlp
import mindspore
import os

# 设置 Token
os.environ["HF_TOKEN"] = "hf_**" 

# 清理离线环境变量，确保联网
if 'HF_DATASETS_OFFLINE' in os.environ: del os.environ['HF_DATASETS_OFFLINE']
if 'TRANSFORMERS_OFFLINE' in os.environ: del os.environ['TRANSFORMERS_OFFLINE']

# 设置随机种子
mindspore.set_seed(42)
print("环境配置完成")

## 3. 数据集加载与预处理

我们将使用 **EuroSAT** 数据集，这是一个基于 Sentinel-2 卫星图像的土地利用和土地覆盖分类数据集，包含 10 个类别（如森林、河流、高速公路等）。由于 EuroSAT 只有 'train' 分割，我们需要将其划分为训练集和验证集。同时，我们需要构建标签 ID 到标签名称的映射字典。

In [ ]:
from mindnlp.dataset import load_dataset
# 加载 EuroSAT 数据
dataset = load_dataset("jonathan-roberts1/EuroSAT", split="train")

# 定义标签映射
labels = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial", 
          "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# 切分数据集
train_dataset, eval_dataset = dataset.split([0.9, 0.1])

### 图像预处理

为了适配 Swin Transformer 模型的输入要求并提升模型泛化能力，我们需要对数据进行特定的预处理。

我们使用 `AutoImageProcessor` 自动加载模型对应的配置（如均值、方差、输入尺寸），并定义了两套不同的处理管道：

* **训练集 (Training)**: 引入随机增强策略（随机裁剪、水平翻转），增加数据多样性，防止过拟合。
* **验证集 (Evaluation)**: 仅进行确定性的调整大小和中心裁剪，确保评估结果的一致性。
* **通用处理**: 包括像素归一化 (Normalize) 和维度转换 (HWC 转 CHW)。


In [ ]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindnlp.transformers import AutoImageProcessor

model_checkpoint = "microsoft/swin-tiny-patch4-window7-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)


# 自动获取模型所需的输入尺寸 (适配不同模型配置)
if "shortest_edge" in image_processor.size:
    size = image_processor.size["shortest_edge"]
    crop_size = (size, size)
elif "height" in image_processor.size:
    size = (image_processor.size["height"], image_processor.size["width"])
    crop_size = size
    
print(f"Image Processor Size Configuration: Resize={size}, Crop={crop_size}")

# 训练集增强策略
def get_train_transforms():
    """
    1. RandomResizedCrop: 随机裁剪并缩放 (让模型学会看局部)
    2. RandomHorizontalFlip: 随机水平翻转 (让模型学会左右不变性)
    3. Rescale + Normalize: 归一化 (对应 ToTensor + Normalize)
    """
    return transforms.Compose([
        vision.RandomResizedCrop(crop_size, scale=(0.08, 1.0), ratio=(0.75, 1.333)),
        vision.RandomHorizontalFlip(prob=0.5),
        vision.Rescale(1.0 / 255.0, 0.0), 
        vision.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
        vision.HWC2CHW()
    ])
    
# 验证集增强策略
def get_eval_transforms():
    """
    1. Resize: 调整大小
    2. CenterCrop: 中心裁剪 (保证每次评估输入一致)
    3. Rescale + Normalize: 归一化
    """
    return transforms.Compose([
        vision.Resize(size),
        vision.CenterCrop(crop_size),
        vision.Rescale(1.0 / 255.0, 0.0),
        vision.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
        vision.HWC2CHW()
    ])

# 定义两个处理函数
def train_transform_func(image, label):
    trans = get_train_transforms()
    return trans(image), label

def eval_transform_func(image, label):
    trans = get_eval_transforms()
    return trans(image), label

# 训练集应用增强策略
train_dataset = train_dataset.map(
    operations=train_transform_func, 
    input_columns=["image", "label"], 
    output_columns=["pixel_values", "labels"]
)

# 验证集应用确定性策略
eval_dataset = eval_dataset.map(
    operations=eval_transform_func, 
    input_columns=["image", "label"], 
    output_columns=["pixel_values", "labels"]
)    

### 数据集内存化适配

为了解决 `Trainer` 在处理流式数据集（IterableDataset）时可能出现的兼容性问题（如 `not subscriptable` 或 `datapipes` 缺失报错），同时充分利用服务器的大内存优势（240GB RAM），我们将处理后的数据集一次性加载到内存列表中。

**核心优势：**
* **兼容性**：将 MindSpore 数据集转换为 Python `List`，使其天然支持索引访问（`__getitem__`），完美适配 Trainer 的数据分发逻辑。
* **训练加速**：避免了训练过程中频繁的磁盘 I/O 读取和实时预处理，数据直接从内存获取，极大提升训练吞吐量。

In [ ]:
print("Converting datasets to memory lists (Compatibility Bridge)...")

def ms_dataset_to_list(ms_dataset):
    data_list = []
    iterator = ms_dataset.create_dict_iterator(num_epochs=1, output_numpy=True)
    for item in iterator:
        data_list.append(item)
    return data_list

# 执行转换
train_dataset_list = ms_dataset_to_list(train_dataset)
eval_dataset_list = ms_dataset_to_list(eval_dataset)
print(f"Converted Train Size: {len(train_dataset_list)}")
print(f"Converted Eval Size: {len(eval_dataset_list)}")

## 4. 模型构建

使用 `AutoModelForImageClassification` 加载预训练的 Swin-Tiny 模型。
- `num_labels`: 设置为 10（EuroSAT 的类别数）。
- `ignore_mismatched_sizes=True`: 允许加载时忽略分类头尺寸不匹配的问题（因为预训练模型是 1000 类 ImageNet，我们需要替换为 10 类），使用随机初始化的分类头。

In [ ]:
from mindnlp.transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

print("Model loaded successfully.")

## 5. 模型训练

使用 MindSpore NLP 提供的 `Trainer` 接口进行训练。我们需要定义 `TrainingArguments` 来配置训练参数（学习率、Epoch 数、保存策略等）。

In [ ]:
from mindnlp.transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

# 加载评估指标
metric = evaluate.load("accuracy")

# 定义评估函数
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

BATCH_SIZE = 32
# 配置训练参数
training_args = TrainingArguments(
    output_dir="swin_tiny_eurosat_finetune", # 输出目录
    eval_strategy="epoch",                   # 每个 epoch 评估一次
    save_strategy="epoch",                   # 每个 epoch 保存一次
    learning_rate=5e-5,                      # 学习率
    per_device_train_batch_size=BATCH_SIZE,  # 训练 Batch Size
    per_device_eval_batch_size=BATCH_SIZE,   # 评估 Batch Size
    num_train_epochs=1,                      # 训练轮次
    weight_decay=0.01,                       # 权重衰减
    load_best_model_at_end=True,             # 训练结束后加载最佳模型
    metric_for_best_model="accuracy",        # 以准确率作为最佳模型指标
    logging_steps=10,                        # 每 10 步打印日志
    save_total_limit=2                       # 最多保存 2 个 Checkpoint
)

# 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_list,
    eval_dataset=eval_dataset_list,
    compute_metrics=compute_metrics,
)

# 开始训练
trainer.train()

## 6. 模型评估与保存

训练完成后，我们在验证集上进行最终评估，并保存微调后的模型。

In [ ]:
# 在验证集上评估
eval_metrics = trainer.evaluate()
print(f"Evaluation Metrics: {eval_metrics}")

# 保存最终模型
trainer.save_model("output/swin-tiny-eurosat")
print("Model saved to output/swin-tiny-eurosat")

## 7. 推理演示

使用微调后的模型对一张新的网络图片进行分类预测。我们将展示如何使用 Pipeline API 快速进行推理。

In [ ]:
from PIL import Image
import requests
from mindnlp.transformers import pipeline


# 下载一张测试图片（不在训练集中）
url = 'https://huggingface.co/nielsr/convnext-tiny-finetuned-eurostat/resolve/main/forest.png'
try:
    image = Image.open(requests.get(url, stream=True).raw)
except:
    # 如果网络无法访问，创建一个随机图片用于演示代码跑通
    print("Network error, using dummy image.")
    image = Image.new('RGB', (224, 224), color='green')

print(f"Test Image Info: {image.format}, Size: {image.size}")

# 创建推理 Pipeline
# task: 图像分类
# model: 刚才训练好的模型对象
# image_processor: 预处理配置
pipe = pipeline("image-classification", model=model, image_processor=image_processor)

# 执行预测
result = pipe(image)

# 打印结果
print("\nPrediction Result:")
for res in result:
    print(f"Label: {res['label']}, Score: {res['score']:.4f}")

print(f"Test Image Info: {image.format}, Size: {image.size}")  